# hy-rect: dual-basis NQS in the sign-problem-full regime, judged against internal certificates

**Campaign 2026-08-20, branch `feat/dual-hy-unblock`.** The `dual_basis` assert against
`hy != 0` was a scoping guard, not physics ($H\sigma_y H = -\sigma_y$ under the global
Hadamard, so $h_y$ is frame-covariant with a sign flip). This notebook is the first
production campaign in the unblocked regime: a $h_x \in \{0.2, 0.6\} \times h_z \in
\{0.1, 0.15\}$ rectangle at L=4 OBC (the same rectangle as `tune_rect_summary.ipynb`),
winner architecture (dual $\cdot$ nh(4$\to$8) $\to$ inv(8,8) $\cdot$ kernel=3 $\cdot$
dt=0.02), three arms:

- **A** — real (float64) ansatz at $h_y=0$: the original tune-rect baseline.
- **C** — complex (complex128) ansatz at $h_y=0$ via `--force_complex`: isolates the cost
  of the complex *parameterization* alone on a still-stoquastic target.
- **D** — complex ansatz at $h_y=0.2$: genuinely sign-full ($H\sigma_y H=-\sigma_y$
  breaks stoquasticity). QMC cannot reach this point (sign problem), so D is judged by
  internal certificates instead: the variational concavity bound $E(h_y) < E(0)$, a
  Hellmann-Feynman cross-check against the $\langle\sigma_y\rangle$ estimator, a
  time-reversal (TR) pair $E(h_y){=}E(-h_y)$, and seed reproducibility.

**Headline (see BLOG 2026-08-20 for the full write-up):** every internal metric passes —
C tracks QMC as well as A (+1.2…+2.8$\sigma$), D sits below the $h_y=0$ QMC band at
every point ($\Delta E = -0.155\ldots-0.246$), Hellmann-Feynman agrees to 0.13%, the TR
pair agrees to 0.93$\sigma$ — at a measured Vscore cost of ~10–30$\times$ for the genuine
sign structure. Everything below reads committed JSONs — no NetKet required.


In [ ]:
# ---- 1 · CONFIG: knobs, paths, arm selectors (single source -- nothing else in this
# notebook hardcodes a path or a style choice) --------------------------------------
import glob, json, math, os, sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = (os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
        if os.getcwd().endswith(os.path.join("analysis", "notebooks")) else os.getcwd())

POINTS = [(0.2, 0.1), (0.6, 0.1), (0.2, 0.15), (0.6, 0.15)]   # the L=4 rectangle
N_SPIN = 144                                                    # L=4 OBC edge count
PCOL = {pt: plt.cm.plasma(0.15 + 0.25 * i) for i, pt in enumerate(POINTS)}
MK = {(0.2, 0.1): "o", (0.6, 0.1): "s", (0.2, 0.15): "^", (0.6, 0.15): "D"}

# Arm file-stem templates ("{x}"/"{z}" are %g-formatted hx/hz -- matches tc3d.train's
# auto run-name). Arm selectors live here ONLY.
ARM = {
    "A": "results/tune_rect/stage1_hx{x}_hz{z}/gridinv_dual_L4_OBC_hx{x}_hz{z}_n2x4_nh4-8_inv8-8_k3_dt0.02",
    "C": "results/hy_rect_L4/gridinv_dual_L4_OBC_hx{x}_hz{z}_n2x4_nh4-8_inv8-8_k3",
    "D": "results/hy_rect_L4/gridinv_dual_L4_OBC_hx{x}_hz{z}_hy0.2_n2x4_nh4-8_inv8-8_k3",
}
ARM_LABEL = {"A": "A: real, $h_y$=0 (tune-rect baseline)",
             "C": "C: complex, $h_y$=0 (--force_complex)",
             "D": "D: complex, $h_y$=0.2 (sign-full)"}
ARM_STYLE = {"A": dict(ls="--", lw=1.3), "C": dict(ls="-", lw=1.3), "D": dict(ls="-", lw=1.7)}

def openax(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

def stem(arm, hx, hz):
    return os.path.join(ROOT, ARM[arm].format(x=f"{hx:g}", z=f"{hz:g}"))

def load_final(arm, hx, hz):
    """{stem}.json -- the completed run: config/observables/curve/n_rollbacks."""
    p = stem(arm, hx, hz) + ".json"
    return json.load(open(p)) if os.path.exists(p) else None

def load_curve(arm, hx, hz):
    """Learning curve dict. Arm A's completed .json carries 'curve' inline (tune-rect
    convention: no separate .curve.json for architecture-tuning runs); arms C/D ship
    the periodic {stem}.curve.json checkpoint instead (same 'curve' sub-dict/schema --
    per-run commit policy: curves are committed here because they ARE the figure input)."""
    p = stem(arm, hx, hz) + (".json" if arm == "A" else ".curve.json")
    if not os.path.exists(p):
        return None
    return json.load(open(p))["curve"]

FIGS = os.path.join(ROOT, "analysis", "figs")
os.makedirs(FIGS, exist_ok=True)
print("ROOT =", ROOT, "| figures ->", FIGS)
print("points:", POINTS)


## 2 · Learning curves per point: arms A (real, $h_y$=0), C (complex, $h_y$=0), D (complex, $h_y$=0.2)

In [ ]:
# ---- 2a: A vs C -- convergence to the (shared) hy=0 QMC reference ------------------
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for ax, (hx, hz) in zip(axes.ravel(), POINTS):
    for arm in ("A", "C"):
        cv = load_curve(arm, hx, hz)
        if cv is None:
            continue
        step = np.asarray(cv["step"], float)
        dref = np.asarray([v if v is not None else np.nan for v in cv["dE_ref"]], float)
        m = np.isfinite(dref) & (dref != 0)
        ax.plot(step[m], np.abs(dref[m]), color=PCOL[(hx, hz)], label=ARM_LABEL[arm],
                **ARM_STYLE[arm])
    ax.set_yscale("log"); openax(ax)
    ax.set_title(f"($h_x$, $h_z$) = ({hx}, {hz})", fontsize=10)
for ax in axes[1]: ax.set_xlabel("step")
for ax in axes[:, 0]: ax.set_ylabel(r"$|E - E_{\rm QMC}|$")
axes[0, 0].legend(frameon=False, fontsize=9)
plt.suptitle("Arms A vs C: convergence to the QMC reference (both target the same $h_y$=0 point)")
plt.tight_layout()
# plt.savefig(os.path.join(FIGS, "hy_rect_learning_curves_AC.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ---- 2b: D -- plain E vs step against the hy=0 QMC band (D must sit BELOW it) -----
# The QMC band (ref_sig ~ 0.02-0.04) is too thin to see against the full descent from
# the cold-init energy, so each panel gets a tail inset (house convention, cf. §4b/4c
# of tune_rect_summary.ipynb) zoomed on the last N_TAIL steps where the band is visible.
N_TAIL = 60
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for ax, (hx, hz) in zip(axes.ravel(), POINTS):
    cv = load_curve("D", hx, hz)
    dD = load_final("D", hx, hz)
    obs = dD["observables"]
    ref_E, ref_sig = obs["ref_E"], obs["ref_sig"]
    step = np.asarray(cv["step"], float)
    E = np.asarray(cv["energy"], float)
    c = PCOL[(hx, hz)]
    ax.plot(step, E, color=c, lw=1.6, label="D: $h_y$=0.2")
    ax.axhline(ref_E, color="0.3", ls="--", lw=1.0, label="QMC ref ($h_y$=0)")
    ax.axhspan(ref_E - ref_sig, ref_E + ref_sig, color="0.3", alpha=0.20, lw=0)
    ax.axhspan(ref_E - 3 * ref_sig, ref_E + 3 * ref_sig, color="0.3", alpha=0.08, lw=0)
    openax(ax)
    ax.set_title(f"($h_x$, $h_z$) = ({hx}, {hz})   " + r"$\Delta E$" + f" = {obs['dE_ref']:+.3f}",
                 fontsize=10)

    x0 = max(0, len(step) - N_TAIL)
    axi = ax.inset_axes([0.42, 0.14, 0.53, 0.40])
    axi.plot(step[x0:], E[x0:], color=c, lw=1.2)
    axi.axhline(ref_E, color="0.3", ls="--", lw=0.9)
    axi.axhspan(ref_E - ref_sig, ref_E + ref_sig, color="0.3", alpha=0.20, lw=0)
    axi.axhspan(ref_E - 3 * ref_sig, ref_E + 3 * ref_sig, color="0.3", alpha=0.08, lw=0)
    lo = min(E[x0:].min(), ref_E - 3.5 * ref_sig); hi = max(E[x0:].max(), ref_E + 1.5 * ref_sig)
    pad = 0.15 * (hi - lo)
    axi.set_ylim(lo - pad, hi + pad); axi.set_xlim(step[x0], step[-1])
    axi.set_xticks([]); axi.set_yticks([])
    ax.indicate_inset_zoom(axi, edgecolor="0.4", alpha=0.7, linewidth=0.8)
for ax in axes[1]: ax.set_xlabel("step")
for ax in axes[:, 0]: ax.set_ylabel(r"$E$")
axes[0, 0].legend(frameon=False, fontsize=9)
plt.suptitle("Arm D ($h_y$=0.2): energy sits below the $h_y$=0 QMC band at every point (concavity bound)")
plt.tight_layout()
# plt.savefig(os.path.join(FIGS, "hy_rect_learning_curves_D.png"), dpi=300, bbox_inches="tight")
plt.show()


## 3 · Final-state table

In [ ]:
# ---- 3: per (point, arm) final-state row -------------------------------------------
ROWS = []
for hx, hz in POINTS:
    for arm in ("A", "C", "D"):
        d = load_final(arm, hx, hz)
        if d is None:
            continue
        obs = d["observables"]
        ROWS.append(dict(
            pt=(hx, hz), arm=arm, E0=obs["E0"], E_err=obs["E_err"],
            z=obs.get("dE_ref_sig") if arm in ("A", "C") else None,
            dE=obs.get("dE_ref") if arm == "D" else None,
            Vscore=obs["Vscore"], E_im=obs.get("E_im"),
            n_rollbacks=d.get("n_rollbacks"),
        ))

def fmt(v, spec, dash="—"):
    return dash if v is None else format(v, spec)

lines = ["| point | arm | E0 ± err | z vs QMC (A/C) | ΔE($h_y$=0.2) below ref (D) | Vscore | E_im | n_rollbacks |",
         "|---|---|---|---|---|---|---|---|"]
for r in ROWS:
    lines.append(
        f"| ({r['pt'][0]}, {r['pt'][1]}) | {r['arm']} | "
        f"{r['E0']:.4f} ± {r['E_err']:.4f} | {fmt(r['z'], '+.2f')} | {fmt(r['dE'], '+.4f')} | "
        f"{r['Vscore']:.2e} | {fmt(r['E_im'], '+.4f')} | {fmt(r['n_rollbacks'], 'd')} |")
display(Markdown("\n".join(lines)))

print("Arm C z-scores (order = POINTS):", [round(r["z"], 2) for r in ROWS if r["arm"] == "C"])
print("Arm D dE      (order = POINTS):", [round(r["dE"], 4) for r in ROWS if r["arm"] == "D"])


## 4 · Sign-structure certificates

### 4a · Hellmann-Feynman: $dE/dh_y$ (finite difference) vs $-N\langle\sigma_y\rangle$

$H = \ldots - h_y \sum_i \sigma^y_i \Rightarrow dE/dh_y = -N\langle\sigma_y\rangle$
(Hellmann-Feynman). An independent-estimator cross-check on the `sy_mean` estimator
and the `sgn_y` sign law, at $(h_x,h_z)=(0.2,0.1)$.

In [ ]:
HX0, HZ0 = 0.2, 0.1
HYS = [0.0, 0.1, 0.2, 0.3]
E_HF, sy_HF = [], []
for hy in HYS:
    tag = "" if hy == 0.0 else f"_hy{hy:g}"
    p = os.path.join(ROOT, f"results/hy_rect_L4/gridinv_dual_L4_OBC_hx{HX0:g}_hz{HZ0:g}{tag}_n2x4_nh4-8_inv8-8_k3.json")
    d = json.load(open(p))
    E_HF.append(d["observables"]["E0"])
    sy_HF.append(d["observables"].get("sy_mean"))

fd_slope = (E_HF[3] - E_HF[1]) / (HYS[3] - HYS[1])      # central FD around hy=0.2
minus_N_sy = -N_SPIN * sy_HF[2]
rel_pct = abs(fd_slope - minus_N_sy) / abs(minus_N_sy) * 100

fig, ax = plt.subplots(figsize=(6.2, 4.3))
c0 = PCOL[(HX0, HZ0)]
ax.plot(HYS, E_HF, "-", color=c0, lw=1.2, zorder=1)
ax.scatter(HYS, E_HF, s=70, facecolors="w", edgecolors=c0, linewidths=1.8,
           zorder=2, label="NQS $E(h_y)$ (open)")
tan_x = np.array([0.1, 0.3])
ax.plot(tan_x, E_HF[2] + fd_slope * (tan_x - 0.2), color="0.3", ls="--", lw=1.3,
        label=f"FD slope @0.2 = {fd_slope:.4f}")
openax(ax)
ax.set_xlabel(r"$h_y$"); ax.set_ylabel(r"$E_0$")
ax.set_title(f"Hellmann-Feynman at ($h_x$,$h_z$)=({HX0},{HZ0})")
ax.legend(frameon=False, fontsize=9)
ax.text(0.03, 0.06,
        f"FD slope = {fd_slope:+.4f}\n" + r"$-N\langle\sigma_y\rangle$" + f" = {minus_N_sy:+.4f}"
        f"  ({rel_pct:.2f}% apart)",
        transform=ax.transAxes, fontsize=9, color="0.25")
plt.tight_layout()
# plt.savefig(os.path.join(FIGS, "hy_rect_hellmann_feynman.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"FD slope @0.2 = {fd_slope:.4f}   -N*sy_mean = {minus_N_sy:.4f}   ({rel_pct:.2f}% apart)")


### 4b · Time-reversal pair: $E(+0.2) \stackrel{?}{=} E(-0.2)$

$T \sigma_y T^{-1} = -\sigma_y$ (time reversal, $x$/$z$ fixed) $\Rightarrow H(h_y)$ and
$H(-h_y)$ share the exact spectrum. Both are independent NQS optimizations, so
agreement within a few $\sigma$ is a genuine cross-check of the sign convention.

In [ ]:
dp = load_final("D", HX0, HZ0)
dm = json.load(open(os.path.join(ROOT,
    f"results/hy_rect_L4/gridinv_dual_L4_OBC_hx{HX0:g}_hz{HZ0:g}_hy-0.2_n2x4_nh4-8_inv8-8_k3.json")))
op, om = dp["observables"], dm["observables"]
pull = (op["E0"] - om["E0"]) / math.hypot(op["E_err"], om["E_err"])
sy_sum = op["sy_mean"] + om["sy_mean"]

fig, ax = plt.subplots(figsize=(4.6, 4.3))
vals, errs = [op["E0"], om["E0"]], [op["E_err"], om["E_err"]]
bars = ax.bar([0, 1], vals, yerr=errs, color=[PCOL[(HX0, HZ0)], "0.6"], capsize=4, width=0.55)
lo, hi = min(vals) - 0.15, max(vals) + 0.08
for x, v, e in zip([0, 1], vals, errs):
    ax.text(x, v + e + 0.008 * (hi - lo), f"{v:.4f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks([0, 1]); ax.set_xticklabels([r"$h_y=+0.2$", r"$h_y=-0.2$"])
ax.set_ylim(lo, hi)
openax(ax)
ax.set_ylabel(r"$E_0$")
ax.set_title(f"TR pair at ({HX0},{HZ0}): pull = {pull:+.2f}$\\sigma$\n"
             r"$\langle\sigma_y\rangle_+ + \langle\sigma_y\rangle_-$" + f" = {sy_sum:+.4f}",
             fontsize=10)
plt.tight_layout()
# plt.savefig(os.path.join(FIGS, "hy_rect_TR_pair.png"), dpi=300, bbox_inches="tight")
plt.show()
print(f"TR pull = {pull:+.2f} sigma, sy+ + sy- = {sy_sum:+.4f}")


### 4c · Seed reproducibility at hy=0.2: weak field (0.2, 0.1) vs strong field (0.6, 0.15)

In [ ]:
SEED_PTS = [(0.2, 0.1), (0.6, 0.15)]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, (hx, hz) in zip(axes, SEED_PTS):
    base = stem("D", hx, hz)
    seeds = []
    for suf in ("", "_s1", "_s2"):
        p = base + suf + ".json"
        if not os.path.exists(p):
            continue
        d = json.load(open(p))
        seeds.append((d["config"]["seed"], d["observables"]["E0"], d["observables"]["E_err"]))
    seeds.sort()
    s, E, err = zip(*seeds)
    ax.errorbar(s, E, yerr=err, fmt=MK[(hx, hz)], ms=9, mfc="w",
                color=PCOL[(hx, hz)], capsize=4, ls="none")
    mean = float(np.mean(E))
    ax.axhline(mean, color=PCOL[(hx, hz)], ls=":", lw=1.0, alpha=0.7)
    ax.set_xticks(s); ax.set_xlabel("seed")
    openax(ax)
    ax.set_title(f"({hx},{hz}): spread (max−min) = {max(E) - min(E):.4f}", fontsize=10)
axes[0].set_ylabel(r"$E_0$")
plt.suptitle("Seed reproducibility at $h_y$=0.2 (arm D)")
plt.tight_layout()
# plt.savefig(os.path.join(FIGS, "hy_rect_seed_scatter.png"), dpi=300, bbox_inches="tight")
plt.show()


## 5 · Vscore cost of sign structure

In [ ]:
V = {arm: [] for arm in ("A", "C", "D")}
for hx, hz in POINTS:
    for arm in ("A", "C", "D"):
        d = load_final(arm, hx, hz)
        V[arm].append(d["observables"]["Vscore"] if d else np.nan)

fig, ax = plt.subplots(figsize=(6.8, 4.4))
xpos = {"A": 0, "C": 1, "D": 2}
for i, pt in enumerate(POINTS):
    for arm in ("A", "C", "D"):
        # two-tone: open marker for the still-stoquastic arms (A, C), filled for the
        # genuinely sign-full arm (D) -- same "open vs filled" convention as the
        # QMC/NQS panels in tune_rect_summary.ipynb, repurposed to flag sign structure.
        ax.scatter(xpos[arm] + (i - 1.5) * 0.07, V[arm][i], s=60, marker=MK[pt],
                   facecolors=PCOL[pt] if arm == "D" else "w", edgecolors=PCOL[pt],
                   linewidths=1.5, label=f"{pt}" if arm == "A" else None)
ax.set_yscale("log")
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["A: real, $h_y$=0", "C: complex, $h_y$=0", "D: complex, $h_y$=0.2"])
openax(ax)
ax.set_ylabel("Vscore")
ax.legend(frameon=False, fontsize=8, title="($h_x$, $h_z$)")
ax.set_title("Vscore by arm: complex dtype is free at $h_y$=0 (A$\\approx$C); sign structure (D) costs ~10–30$\\times$")
plt.tight_layout()
# plt.savefig(os.path.join(FIGS, "hy_rect_vscore_cost.png"), dpi=300, bbox_inches="tight")
plt.show()

ratios_DA = [d / a for d, a in zip(V["D"], V["A"])]
ratios_DC = [d / c for d, c in zip(V["D"], V["C"])]
print("Vscore ratio D/A:", [f"{r:.1f}x" for r in ratios_DA])
print("Vscore ratio D/C:", [f"{r:.1f}x" for r in ratios_DC])


## 6 · X1 (primal cross-check) — PENDING, data still chaining

The primal-frame (non-dual) complex cross-check at $(h_x,h_z,h_y)=(0.2,0.1,0.2)$
(`results/hy_rect_L4/gridinv_L4_OBC_hx0.2_hz0.1_hy0.2_n2x4_nh4-8_inv8-8_k3`) hit the
dense-QGT XLA anomaly mid-run (qgt step time 0.9s $\to$ 104s, same signature as the
2026-07-21 $h_y$=0.4 note) and is still AUTO_RESUBMIT-chaining on the cluster — only a
`.curve.json` checkpoint exists so far (170/300 steps at the time of writing), no
completed `.json`. **Rerun this section once X1 lands** and add it as a fourth arm
above (dual-frame runs never show this anomaly — one more reason the dual frame is the
production lane, but X1 is the one direct primal-vs-dual comparison still missing).
